# Training/Forecasting Strategy Figure

Builds the "movie analogy" figure for the paper's Methodology section (`fig:training_strategy`):
4 stacked L1 solar-wind/IMF timeseries spanning the full N+M window (always available, the "audio track"),
and below, a strip of electric potential map frames where the first N=15 are real observed frames
and the last M=7 are blanked out (the "video track" to be predicted).

Uses the real `IonoSequenceDataset` production pipeline (same class used for training) to pull one
genuine sequence from the training split, so the figure reflects actual data, not a synthetic mockup.

In [1]:
import sys, os
sys.path.insert(0, '/users/framunno/projects/ionosphere_diffusion')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import map_coordinates
from matplotlib.patches import Rectangle, FancyArrowPatch

from src.data.dataset import IonoSequenceDataset

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'STIXGeneral'

PAPER_FIGURES_DIR = '/users/framunno/projects/paper_writing/SuperDARN_deep_Learning_Francesco/figures'
CSV_PATH = '/users/framunno/data/ionosphere/l1_to_map_matched_2020_2025.csv'


In [2]:
# Same construction as training_pred.py / training_script_ddp.sh for the CLASSIC config:
# sequence_length = cond_frames (15) + pred_frames (7) = 22, cartesian_transform, output_size=128,
# normalization_type='absolute_max', use_l1_conditions=True, min_center_distance=15 (train split).
N_COND, M_PRED = 15, 7
SEQ_LEN = N_COND + M_PRED

ds = IonoSequenceDataset(
    csv_path=CSV_PATH,
    transform_cond_csv=None,
    sequence_length=SEQ_LEN,
    normalization_type='absolute_max',
    use_l1_conditions=True,
    min_center_distance=15,
    cartesian_transform=True,
    output_size=128,
    only_complete_sequences=True,
    split='valid',
)
print('num complete valid sequences:', len(ds))


Building L1 conditions cache for valid split...


Cached L1 conditions for 1370626 files


Filtered sequences for valid: 25299 -> 18312 (72.4% complete sequences)
num complete valid sequences: 18312


In [3]:
# Rank sequences by geomagnetic activity (mean |Bz| * mean SWV, physical units) using the
# cached L1 conditions dict, so we pick a visually clear two-cell convection pattern
# rather than a random quiet interval.
MAPS_BASE = '/capstor/scratch/cscs/framunno/ionosphere_data/all_maps/'
scores = []
for i, center_idx in enumerate(ds.sequences):
    start_idx = center_idx - ds.sequence_length // 2
    bzs, speeds, ok = [], [], True
    for off in range(ds.sequence_length):
        fidx = start_idx + off
        if fidx < 0 or fidx >= len(ds.all_files):
            ok = False
            break
        fname = os.path.basename(ds.all_files[fidx])
        if fname not in ds.filename_to_conditions:
            ok = False
            break
        if not os.path.exists(MAPS_BASE + fname):
            ok = False
            break
        bx, by, bz, vx = ds.filename_to_conditions[fname]
        bzs.append(bz)
        speeds.append(abs(vx))
    if not ok:
        continue
    score = -np.mean(bzs) * np.mean(speeds)
    scores.append((score, i, center_idx))

scores.sort(reverse=True)
print('top candidates:')
for s in scores[:8]:
    print(s, ds.all_timestamps[s[2]])


top candidates:
(np.float32(26214.812), 17603, 1349951) 2025-11-12 00:54:00
(np.float32(13355.425), 10978, 1149191) 2024-10-10 20:22:00
(np.float32(12603.918), 10974, 1149131) 2024-10-10 18:22:00
(np.float32(10701.292), 10982, 1149251) 2024-10-10 22:22:00
(np.float32(8486.072), 14213, 1270631) 2025-04-16 15:16:00
(np.float32(8482.267), 14212, 1270616) 2025-04-16 14:46:00
(np.float32(8332.626), 14211, 1270601) 2025-04-16 14:16:00
(np.float32(8221.633), 10980, 1149221) 2024-10-10 21:22:00


In [4]:
# Pick a strong, clean candidate (adjust index below after inspecting the printout above if needed)
CHOSEN_RANK = 0
chosen_seq_i = scores[CHOSEN_RANK][1]
print('chosen sequence dataset index:', chosen_seq_i, 'center time:', ds.all_timestamps[scores[CHOSEN_RANK][2]])

data_seq, cond_seq = ds[chosen_seq_i]  # data_seq: (22,1,128,128) in [-1,1], cond_seq: (22,4) in [-1,1]

# Denormalize to physical units. NOTE: ds.reverse_data_normalization() has a bug for the
# 'absolute_max' case (it inverts the OLD forward formula that mapped to [0,1], but the
# actual forward formula in __getitem__ maps to [-1,1]: y = (x+80000)/80000 - 1, i.e. x = 80000*y).
# The paper's actual results figures (paper_2015_storm_figures.ipynb) never use this class
# method either -- they denormalize predictions manually as `pred * NORM_SCALE` (NORM_SCALE=80000),
# which is the correct inverse. We do the same here instead of calling the buggy method.
NORM_SCALE = 80000
data_phys = (data_seq.squeeze(1).numpy() * NORM_SCALE)                           # (22,128,128) Volts
cond_phys = ds.revert_condition_normalization(cond_seq).numpy()                  # (22,4) [bx,by,bz,vx_gsm] -- this one is correct

bx, by, bz, vx_gsm = cond_phys[:,0], cond_phys[:,1], cond_phys[:,2], cond_phys[:,3]
swv = np.abs(vx_gsm)  # speed magnitude

center_idx = ds.sequences[chosen_seq_i]
start_idx = center_idx - ds.sequence_length // 2
times = [ds.all_timestamps[start_idx+off] for off in range(ds.sequence_length)]
print(times[0], '->', times[-1])
print('Bz range:', bz.min(), bz.max(), ' SWV range:', swv.min(), swv.max())


chosen sequence dataset index: 17603 center time: 2025-11-12 00:54:00


2025-11-12 00:32:00 -> 2025-11-12 01:14:00
Bz range: -53.55 -3.630001  SWV range: 449.2 719.0


In [5]:
# Fetch the real per-frame L1->ionosphere propagation delay (delay_min column) for the
# chosen sequence, so the "delay" annotation in the figure uses an actual measured value
# rather than a generic placeholder.
import pandas as pd
df_delay = pd.read_csv(CSV_PATH, usecols=['filename', 'delay_min'])
df_delay['fname'] = df_delay['filename'].apply(os.path.basename)
delay_lookup = dict(zip(df_delay['fname'], df_delay['delay_min']))

seq_filenames = [os.path.basename(ds.all_files[start_idx + off]) for off in range(ds.sequence_length)]
seq_delays = [delay_lookup.get(f, float('nan')) for f in seq_filenames]
mean_delay_min = float(np.nanmean(seq_delays))
print('per-frame delay_min (L1 -> Earth arrival):', [round(d,1) for d in seq_delays])
print('mean delay for this sequence (min):', round(mean_delay_min, 1))


per-frame delay_min (L1 -> Earth arrival): [54.4, 35.2, 34.4, 34.7, 34.3, 34.5, 34.8, 35.0, 35.0, 35.1, 34.0, 34.1, 34.0, 34.5, 34.8, 34.9, 34.6, 34.5, 34.8, 34.7, 35.2, 35.6]
mean delay for this sequence (min): 35.6


In [6]:
# Polar remap helper - identical recipe used for storm2015_frames_grid_*.png (paper_2015_storm_figures.ipynb)
H_IMG, MAX_R = 128, 24
r_i, th_i = np.linspace(0, MAX_R, 200), np.linspace(0, 2*np.pi, 360)
r_grid, theta_grid = np.meshgrid(r_i, th_i)
polar_x, polar_y = r_grid*np.cos(theta_grid), r_grid*np.sin(theta_grid)
col_coords = (polar_x + MAX_R) / (2*MAX_R) * (H_IMG - 1)
row_coords = (polar_y + MAX_R) / (2*MAX_R) * (H_IMG - 1)

def to_polar(frame2d):
    return map_coordinates(frame2d, [row_coords, col_coords], order=1, mode='constant', cval=0)

VMAX = np.abs(data_phys[:N_COND]).max()
print('VMAX for colormap:', VMAX)


VMAX for colormap: 48583.66


In [7]:
fig = plt.figure(figsize=(17, 9.6))

# Single outer grid: 4 timeseries rows + 1 thin connector row + 1 frame-strip row, all sharing
# the exact same left/right extent so the two blocks line up automatically.
gs_outer = gridspec.GridSpec(
    7, 1, height_ratios=[1, 1, 1, 1, 0.35, 2.4, 1.1], hspace=0.0,
    left=0.085, right=0.985, top=0.94, bottom=0.13, figure=fig,
)

series = [swv, bx, by, bz]
row_labels = ['SWV\n[km/s]', r'$B_x$' + '\n[nT]', r'$B_y$' + '\n[nT]', r'$B_z$' + '\n[nT]']
colors = ['#2b2d42', '#3a6ea5', '#2e8b57', '#c1272d']
x = np.arange(SEQ_LEN)

ts_axes = []
for k in range(4):
    ax = fig.add_subplot(gs_outer[k])
    ts_axes.append(ax)
    ax.plot(x, series[k], color=colors[k], lw=2.3, solid_capstyle='round', zorder=3)
    ax.axvline(N_COND - 1, color='0.3', ls=(0, (4, 2)), lw=1.3, zorder=2)
    ax.set_ylabel(row_labels[k], fontsize=14.5, rotation=0, ha='right', va='center', labelpad=14,
                   fontfamily='STIXGeneral')
    ax.set_xlim(-0.5, SEQ_LEN - 0.5)
    ax.set_xticks([])
    ax.margins(y=0.22)
    ax.tick_params(labelsize=11, length=0)
    # a visible top border on every row (except the first) acts as a divider line between
    # the stacked timeseries panels, so they read as distinct rows instead of one blurred block
    ax.spines['top'].set_visible(k > 0)
    if k > 0:
        ax.spines['top'].set_color('0.85')
        ax.spines['top'].set_linewidth(1.0)
    for spine in ['right', 'bottom', 'left']:
        ax.spines[spine].set_visible(False)
    ax.set_facecolor('none')

ts_axes[0].text(N_COND - 1, 1.0, r'$t_0$', ha='center', va='bottom', fontsize=15,
                 transform=ts_axes[0].get_xaxis_transform(), zorder=4, fontfamily='STIXGeneral')

# thin connector row: a single dashed line bridging the timeseries block and the frame strip
ax_conn = fig.add_subplot(gs_outer[4])
ax_conn.set_xlim(-0.5, SEQ_LEN - 0.5)
ax_conn.set_ylim(0, 1)
ax_conn.axis('off')
ax_conn.plot([N_COND - 1, N_COND - 1], [0, 1], color='0.3', ls=(0, (4, 2)), lw=1.3, zorder=2)

# ---- frame strip: real frames for t_-N..t_0, blanked for t_1..t_M ----
gs_frames = gridspec.GridSpecFromSubplotSpec(1, SEQ_LEN, subplot_spec=gs_outer[5], wspace=0.10)
frame_axes = []
for i in range(SEQ_LEN):
    ax = fig.add_subplot(gs_frames[i], projection='polar')
    frame_axes.append(ax)
    ax.set_ylim(0, MAX_R)
    ax.grid(False)
    if i < N_COND:
        ax.pcolormesh(theta_grid, r_grid, to_polar(data_phys[i]), shading='auto',
                       cmap='coolwarm', vmin=-VMAX, vmax=VMAX, zorder=1)
        ax.spines['polar'].set_linewidth(1.0)
        ax.spines['polar'].set_color('0.5')
    else:
        ax.set_facecolor('#fafafa')
        ax.text(0, 0, '?', ha='center', va='center', fontsize=19, color='#b5b5b5',
                 fontweight='bold', zorder=2, fontfamily='STIXGeneral')
        ax.spines['polar'].set_linewidth(1.3)
        ax.spines['polar'].set_color('#c9c9c9')
        ax.spines['polar'].set_linestyle((0, (3, 2)))
    ax.set_theta_zero_location('S')
    ax.set_theta_direction(1)
    ax.set_xticks([])
    ax.set_yticks([])

fig.canvas.draw()

# ---- shaded band marking the "future" (to-predict) region -- ONLY over the frame strip row,
# since the L1 conditions are observed for the FULL window (they condition the model; they are
# never the thing being predicted, so the shading must not bleed into the timeseries block). ----
x_t0_fig = (frame_axes[N_COND - 1].get_position().x1 + frame_axes[N_COND].get_position().x0) / 2
x_right_fig = frame_axes[-1].get_position().x1
y_top = frame_axes[0].get_position().y1 + 0.02
y_bot = frame_axes[0].get_position().y0 - 0.01
band = Rectangle((x_t0_fig, y_bot), x_right_fig - x_t0_fig, y_top - y_bot,
                  transform=fig.transFigure, facecolor='#f2a154', alpha=0.14, zorder=0,
                  edgecolor='none')
fig.add_artist(band)

# ---- OBSERVED / TO PREDICT labels directly above the frame strip, aligned to the actual axes ----
x_obs_mid = (frame_axes[0].get_position().x0 + frame_axes[N_COND - 1].get_position().x1) / 2
x_pred_mid = (frame_axes[N_COND].get_position().x0 + frame_axes[-1].get_position().x1) / 2
y_label = frame_axes[0].get_position().y1 + 0.015
fig.text(x_obs_mid, y_label, 'OBSERVED', ha='center', fontsize=13, fontweight='bold', color='#3a6ea5',
          fontfamily='STIXGeneral')
fig.text(x_pred_mid, y_label, 'TO PREDICT', ha='center', fontsize=13, fontweight='bold', color='#c1662d',
          fontfamily='STIXGeneral')

# ---- N/M captions below the frame strip, aligned to the actual axes ----
y_cap = frame_axes[0].get_position().y0 - 0.045
fig.text(x_obs_mid, y_cap, r'$N=15$ past frames (30 min)', ha='center', fontsize=13.5, color='0.25',
          fontfamily='STIXGeneral')
fig.text(x_pred_mid, y_cap, r'$M=7$ future frames (14 min)', ha='center', fontsize=13.5, color='0.25',
          fontfamily='STIXGeneral')

fig.text(0.085, frame_axes[0].get_position().y1 + 0.06, 'Ionospheric electric potential maps',
          fontsize=15, ha='left', color='0.15', fontfamily='STIXGeneral')

# ---- ground-truth reveal row: directly below the "to predict" section, show what actually
# happened for those M=7 frames (real data, not the model's prediction), so the reader can see
# what the blanked-out "?" spheres above correspond to. Same 22-column grid so the 7 real
# columns land exactly under their corresponding "?" placeholder above.
gs_gt = gridspec.GridSpecFromSubplotSpec(1, SEQ_LEN, subplot_spec=gs_outer[6], wspace=0.10)
gt_axes = []
for i in range(N_COND, SEQ_LEN):
    ax = fig.add_subplot(gs_gt[i], projection='polar')
    gt_axes.append(ax)
    ax.set_ylim(0, MAX_R)
    ax.grid(False)
    ax.pcolormesh(theta_grid, r_grid, to_polar(data_phys[i]), shading='auto',
                   cmap='coolwarm', vmin=-VMAX, vmax=VMAX, zorder=1)
    ax.spines['polar'].set_linewidth(1.0)
    ax.spines['polar'].set_color('0.5')
    ax.set_theta_zero_location('S')
    ax.set_theta_direction(1)
    ax.set_xticks([])
    ax.set_yticks([])

fig.canvas.draw()
y_gt_label = gt_axes[0].get_position().y1 + 0.012
fig.text(x_pred_mid, y_gt_label, 'GROUND TRUTH', ha='center', fontsize=13, fontweight='bold',
          color='0.2', fontfamily='STIXGeneral')

plt.savefig(f'{PAPER_FIGURES_DIR}/methodology_training_strategy.png', dpi=300, bbox_inches='tight')
plt.show()
